# Building a baseline qodec and instruction sets

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How does a qodec organize logical and physical instruction sets around a stabilizer code?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Explain how a `qodec` pairs each protocol layer with an `InstructionSet` (`layer.instruction_set`).
- Build a two-layer baseline `qodec` from a stabilizer code with `ec.build_qodec` and inspect omitted instructions.
- Read and author formal instruction contracts (`Stabilize`, `Clifford`, `Pauli`, `Observe`) in a `qodec` instruction set.

## Setting up

The cell below recreates the `C4` code defined in *Defining and Profiling a Stabilizer Code*.

In [ ]:
import qodec
from qdk import ec
from _unit import make_c4

c4 = make_c4()

## From a code to a layered qodec

In the previous unit, `qodec.Code` defined which four-qubit states belong to `C4`, but a code alone does not specify any operations. To compute on encoded states, a **qodec** (`qodec.Qodec`) organizes a protocol as a stack of layers, where **each layer is defined by an instruction set** (`qodec.InstructionSet`) listing the operations available at that level of abstraction.

Calling `ec.build_qodec` builds a complete two-layer `qodec`—including both instruction sets and the gadgets connecting them—directly from `c4`:
- **Layer 0 (`logical_layer.instruction_set`):** declares the logical instruction set over `C4` blocks and pairs each instruction with a gadget.
- **Layer 1 (`physical_layer.instruction_set`):** declares the target physical instruction set (`R`, `H`, `CX`, `CZ`, `M`) with no gadgets beneath it.

`build_qodec` currently ships minimal CSS starter strategies—such as `flagged-css/v1` *(default)* and unflagged `bare-css/v1`—intended as convenient starting points for bootstrapping and inspecting protocols (these built-in strategies are expected to evolve in future releases). We start with `bare-css/v1` here so we can examine unflagged baseline circuits and pinpoint exactly where circuit-level faults arise.

Each CSS starter strategy attempts seven candidate instructions (`prepare_z_all`, `prepare_x_all`, `syndrome`, `measure_z_all`, `measure_x_all`, `h_all`, and `cx_all`), where `_all` means the instruction acts on every logical qubit in the block. Passing `strict=False` retains every supported instruction and records skipped candidates in `protocol.metadata`:

In [ ]:
protocol = ec.build_qodec(c4, strategy="bare-css/v1", strict=False)
logical_layer, physical_layer = protocol.layers

print("Logical instructions :", list(logical_layer.instruction_set.instructions))
print("Physical instructions:", list(physical_layer.instruction_set.instructions))
print("Omitted candidates   :", protocol.metadata["qdk.ec"]["build"]["omitted"])

Why was `h_all` omitted? On `C4`, transversal Hadamard $H^{\otimes 4}$ maps $\bar{X}_0 = X_0 X_1$ to $Z_0 Z_1 = \bar{Z}_1$ rather than $\bar{Z}_0 = Z_0 Z_2$, swapping the two logical qubits instead of implementing an un-swapped `h_all` (the Capstone adds this operation under its true contract).

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;What happens if you build a C4 baseline qodec with the default strict=True setting?</summary>

<div style="padding:0.1em 1em;">

`build_qodec` raises an error because transversal Hadamard on `C4` swaps the two logical qubits instead of implementing an un-swapped `h_all` contract. Setting `strict=False` retains the six valid instructions and records `h_all` under `protocol.metadata['qdk.ec']['build']['omitted']`.

</div>

</details></div>

## Formal instruction actions

Each `qodec.Instruction` declares its mathematical contract through `action` primitives—completely independent of whatever circuit a gadget uses to implement it. Let's inspect four concrete instructions from `C4`'s logical instruction set first:

In [ ]:
instructions = logical_layer.instruction_set.instructions

for name in ["prepare_z_all", "cx_all", "measure_z_all", "syndrome"]:
    instr = instructions[name]
    print(f"=== {name} ===")
    print(f"  in={list(instr.inputs)}  out={list(instr.outputs)}")
    print(f"  action={list(instr.action)}")

print("\nYAML manifest preview:")
print("\n".join(protocol.dumps().splitlines()[:18]))

Comparing the printed `action` lists above illustrates the four `qodec.actions` primitives:

| Action type | Meaning | Example in `C4` |
| :--- | :--- | :--- |
| `Stabilize` | Prepares output state stabilized by the listed logical Paulis | `prepare_z_all` stabilizes `Z_0, Z_1` ($\vert 00\rangle_L$) |
| `Clifford` | Maps input logical Pauli generators to output logical Paulis (unlisted generators map to themselves) | `cx_all` maps `X_0 -> X_0 X_2`, `Z_2 -> Z_0 Z_2`, ... |
| `Pauli` | Applies a logical Pauli operator up to frame sign | Single-block logical Pauli flip |
| `Observe` | Measures commuting logical Pauli observables in order | `measure_z_all` observes `Z_0`, then `Z_1` |

Multi-block instructions flatten logical qubit indices across operands in order: for `cx_all(control, target)` on two `C4` blocks, logical indices `0, 1` belong to `control` and `2, 3` belong to `target`.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does the syndrome instruction have an empty action list?</summary>

<div style="padding:0.1em 1em;">

Syndrome extraction is a logical identity on the encoded block: it takes one `C4` block in and returns one `C4` block out without preparing, rotating, or destructively observing any logical qubit.

</div>

</details></div>

## Exercise: Author a logical SWAP instruction contract

Construct and return a `qodec.Instruction` named `'swap_all'` that takes one `'C4'` `BlockOperand` input, produces one `'C4'` `BlockOperand` output, and declares a `Clifford` action swapping logical qubits `0` and `1` (`X_0 <-> X_1` and `Z_0 <-> Z_1`). Remember that `Clifford({...})` treats any omitted generator as mapping to itself, so all four swapped generators must be listed explicitly.

In [ ]:
import qodec
from qodec.actions import Clifford
from qodec.instructions import BlockOperand
from _unit import exercise


@exercise
def build_logical_swap_instruction() -> qodec.Instruction:
    # Return a qodec.Instruction('swap_all', ...) that swaps logical qubits 0 and 1 on C4
    return ...

**Hint**

Pass `mnemonic='swap_all'`, `inputs=[BlockOperand('C4')]`, `outputs=[BlockOperand('C4')]`, and `action=[Clifford({'X_0': 'X_1', 'Z_0': 'Z_1', 'X_1': 'X_0', 'Z_1': 'Z_0'})]` to `qodec.Instruction`.

In [ ]:
import qodec
from qodec.actions import Clifford
from qodec.instructions import BlockOperand
from _unit import exercise


@exercise
def build_logical_swap_instruction() -> qodec.Instruction:
    return qodec.Instruction(
        mnemonic="swap_all",
        inputs=[BlockOperand("C4")],
        outputs=[BlockOperand("C4")],
        action=[Clifford({"X_0": "X_1", "Z_0": "Z_1", "X_1": "X_0", "Z_1": "Z_0"})],
    )

A single `C4` block encodes logical qubits `0` and `1`. Listing all four generator images (`X_0 -> X_1`, `Z_0 -> Z_1`, `X_1 -> X_0`, `Z_1 -> Z_0`) preserves pairwise anticommutation and specifies the exact logical SWAP contract independent of whether a gadget implements it via physical qubit relabeling, SWAP gates, or three CNOTs.

## Further reading

- [Build a baseline qodec ↗](https://learn.microsoft.com/en-us/azure/quantum/develop-validate-qec-protocol#build-a-baseline-qodec)
- [Instruction Set concepts in qodec ↗](https://github.com/microsoft/qdk-ec/blob/main/qodec/docs/concepts/instruction-set.md)
- [The qodec model ↗](https://github.com/microsoft/qdk-ec/blob/main/qodec/docs/concepts/model.md)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../01-stabilizer-codes/stabilizer_codes.workbook.ipynb" title="Previous unit: Defining and Profiling a Stabilizer Code" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Defining and Profiling a Stabilizer Code</span></span></a><a href="../03-gadget-anatomy/gadget_anatomy.workbook.ipynb" title="Next unit: Inspecting Gadgets, Checks, and Readouts" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Inspecting Gadgets, Checks, and Readouts</span></span><span aria-hidden="true">&#8250;</span></a></div>